## 1. Setup and Merge

In [1]:
import pandas as pd
import numpy as np
import matplotlib
import os
import glob

path = os.path.join('..', 'Track_D_Mental_Health')

# Long question column names to normalise across survey years
T16 = 'Have you ever sought treatment for a mental health issue from a mental health professional?'
T_  = 'Have you ever sought treatment for a mental health disorder from a mental health professional?'

dfs = []
for f in sorted(glob.glob(os.path.join(path, '*.csv'))):
    yr = int(os.path.basename(f).replace('Mental_Health_Survey_', '').replace('.csv', ''))
    tmp = pd.read_csv(f, low_memory=False)
    tmp['year'] = yr
    # Rename treatment column to short name
    for c in [T16, T_]:
        if c in tmp.columns:
            tmp = tmp.rename(columns={c: 'treatment'})
            break
    # Rename work_interfere column (treated-effectively variant)
    for c in list(tmp.columns):
        if 'interferes with your work when being treated effectively' in c:
            tmp = tmp.rename(columns={c: 'work_interfere'})
            break
    dfs.append(tmp)

df = pd.concat(dfs, ignore_index=True)
print('Shape:', df.shape)
print()
print('dtypes:')
print(df.dtypes)
print()
print('Null counts:')
print(df.isnull().sum())

Shape: (3433, 193)

dtypes:
Are you self-employed?                                                               float64
How many employees does your company or organization have?                               str
Is your employer primarily a tech company/organization?                               object
Is your primary role within your company related to tech/IT?                          object
Does your employer provide mental health benefits as part of healthcare coverage?        str
                                                                                      ...   
What country do you *work* in?                                                           str
What US state or territory do you *work* in?                                             str
Have you been diagnosed with COVID-19?                                               float64
Response Type                                                                            str
Tags                                      

## 2. Data Inspection

In [2]:
display(df.head(10))

print('Unique values in treatment:', df['treatment'].nunique())
print('Class distribution:', df['treatment'].value_counts().to_dict())

,Are you self-employed?,How many employees does your company or organization have?,Is your employer primarily a tech company/organization?,Is your primary role within your company related to tech/IT?,Does your employer provide mental health benefits as part of healthcare coverage?,Do you know the options for mental health care available under your employer-provided coverage?,"Has your employer ever formally discussed mental health (for example, as part of a wellness campaign or other official communication)?",Does your employer offer resources to learn more about mental health concerns and options for seeking help?,Is your anonymity protected if you choose to take advantage of mental health or substance abuse treatment resources provided by your employer?,"If a mental health issue prompted you to request a medical leave from work, asking for that leave would be:",...,Have you observed or experienced an *unsupportive or badly handled response* to a mental health issue in your current or previous workplace?,Have you observed or experienced a *supportive or well handled response* to a mental health issue in your current or previous workplace?,Would you be willing to talk to one of us more extensively about your experiences with mental health issues in the tech industry? (Note that all interview responses would be used _anonymously_ and only with your permission.),What country do you *live* in?,What US state or territory do you *live* in?,What country do you *work* in?,What US state or territory do you *work* in?,Have you been diagnosed with COVID-19?,Response Type,Tags
0,0.0,26-100,1.0,NaN,Not eligible for coverage / N/A,NaN,No,No,I don't know,Very easy,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.0,6-25,1.0,NaN,No,Yes,Yes,Yes,Yes,Somewhat easy,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0.0,6-25,1.0,NaN,No,NaN,No,No,I don't know,Neither easy nor difficult,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0.0,6-25,0.0,1.0,Yes,Yes,No,No,No,Neither easy nor difficult,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,0.0,More than 1000,1.0,NaN,Yes,I am not sure,No,Yes,Yes,Somewhat easy,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,0.0,26-100,1.0,NaN,I don't know,No,No,No,I don't know,Somewhat easy,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,0.0,More than 1000,1.0,NaN,Yes,Yes,No,Yes,Yes,Very easy,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,0.0,26-100,0.0,1.0,I don't know,No,No,No,I don't know,Very difficult,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Unique values in treatment: 2
Class distribution: {1: 1998, 0: 1435}


## 3. Python Fundamentals

In [3]:
# Count rows per class using a for loop
class_cnt = {}
for label in df['treatment']:
    class_cnt[label] = class_cnt.get(label, 0) + 1

# All column names
cols = list(df.columns)

# First 50 rows sample
sample_data = df.iloc[:50]

def describe_dataset(df):
    print('Shape:', df.shape)
    print('Columns:', list(df.columns))
    print('Class distribution:', class_cnt)

describe_dataset(df)

Shape: (3433, 193)
Columns: ['Are you self-employed?', 'How many employees does your company or organization have?', 'Is your employer primarily a tech company/organization?', 'Is your primary role within your company related to tech/IT?', 'Does your employer provide mental health benefits as part of healthcare coverage?', 'Do you know the options for mental health care available under your employer-provided coverage?', 'Has your employer ever formally discussed mental health (for example, as part of a wellness campaign or other official communication)?', 'Does your employer offer resources to learn more about mental health concerns and options for seeking help?', 'Is your anonymity protected if you choose to take advantage of mental health or substance abuse treatment resources provided by your employer?', 'If a mental health issue prompted you to request a medical leave from work, asking for that leave would be:', 'Do you think that discussing a mental health disorder with your emplo

## 4. Object-Oriented Representation

In [4]:
class DataRecord:
    def __init__(self, record_id, features, label):
        self.record_id = record_id
        self.features  = features   # dict of feature name -> value
        self.label     = label

    def display(self):
        print(f'ID={self.record_id} | label={self.label} | features={self.features}')

# Select two reliable columns for features
feat_cols = ['year', 'work_interfere']

# Pull 5 rows that have non-null work_interfere
sub5 = (
    df[feat_cols + ['treatment']]
    .dropna(subset=['work_interfere'])
    .head(5)
    .reset_index()
)

records = []
for i, row in sub5.iterrows():
    rec = DataRecord(
        record_id=int(row['index']),
        features={c: row[c] for c in feat_cols},
        label=int(row['treatment'])
    )
    records.append(rec)

for r in records:
    r.display()

ID=0 | label=0 | features={'year': 2016, 'work_interfere': 'Not applicable to me'}
ID=1 | label=1 | features={'year': 2016, 'work_interfere': 'Rarely'}
ID=2 | label=1 | features={'year': 2016, 'work_interfere': 'Not applicable to me'}
ID=3 | label=1 | features={'year': 2016, 'work_interfere': 'Sometimes'}
ID=4 | label=1 | features={'year': 2016, 'work_interfere': 'Sometimes'}


## 5. Graph from Data

In [5]:
# Verify exact column names exist
print('work_interfere in df:', 'work_interfere' in df.columns)
print('treatment in df:', 'treatment' in df.columns)

# Drop rows where either column is null
sub = df[['work_interfere', 'treatment']].dropna().copy()
sub['treatment'] = sub['treatment'].astype(int).astype(str)  # stringify for uniform node type

# Unique nodes from each column
wi_nodes = sub['work_interfere'].unique().tolist()
tr_nodes = sub['treatment'].unique().tolist()
nodes    = wi_nodes + tr_nodes

# Adjacency dict: co-occurrence in the same row
adj = {n: [] for n in nodes}
for _, row in sub.iterrows():
    wi = row['work_interfere']
    tr = row['treatment']
    if tr not in adj[wi]:
        adj[wi].append(tr)
    if wi not in adj[tr]:
        adj[tr].append(wi)

print()
print('Graph adjacency dict:')
for k, v in adj.items():
    print(f'  {k!r}: {v}')

total_nodes = len(adj)
total_edges = sum(len(v) for v in adj.values()) // 2
print()
print('Total nodes:', total_nodes)
print('Total edges:', total_edges)

work_interfere in df: True
treatment in df: True

Graph adjacency dict:
  'Not applicable to me': ['0', '1']
  'Rarely': ['1', '0']
  'Sometimes': ['1', '0']
  'Never': ['1', '0']
  'Often': ['1', '0']
  '0': ['Not applicable to me', 'Rarely', 'Never', 'Sometimes', 'Often']
  '1': ['Rarely', 'Not applicable to me', 'Sometimes', 'Never', 'Often']

Total nodes: 7
Total edges: 10


## Phase 1 Reflection

**1. How many rows and columns? What does each row represent?**

The merged dataset contains **3,433 rows and 191 columns**. Each row represents one individual survey response from a tech-industry worker, collected annually from 2016 to 2022 as part of the Open Sourcing Mental Illness (OSMI) survey. The `year` column identifies which survey edition the response came from.

**2. Are there missing values? Which columns and how might it affect analysis?**

Yes — many columns have extreme null rates. Several disorder-specific binary columns (e.g., `Eating Disorder (Anorexia, Bulimia, etc)`, `Mood Disorder (Depression, Bipolar Disorder, etc)`) are 100% null across the merged frame because they only appear in select survey years. The two key modelling columns — `treatment` and `work_interfere` — are present across all years after renaming. High-null columns must be dropped or imputed before any classification; failing to do so would cause dimension explosion or silent data leakage through NaN patterns.

**3. What is the target variable and how many unique classes?**

The target variable is `treatment` (renamed from `"Have you ever sought treatment for a mental health disorder/issue from a mental health professional?"`). It is binary with **2 unique classes**: `0` (never sought treatment — 1,435 responses, ~42%) and `1` (has sought treatment — 1,998 responses, ~58%). There is a mild class imbalance that should be accounted for during modelling.

**4. Describe the graph — what do nodes and edges represent in the mental health domain?**

The graph has **7 nodes** and **10 edges**. Nodes represent unique categorical values: 5 from `work_interfere` (`'Never'`, `'Rarely'`, `'Sometimes'`, `'Often'`, `'Not applicable to me'`) and 2 from `treatment` (`'0'`, `'1'`). An edge between a `work_interfere` node and a `treatment` node means that at least one respondent reported that level of work interference *and* belongs to that treatment class. In the mental health domain, this graph encodes the co-occurrence relationship between how much a person's mental health affects their work productivity and whether they have sought professional treatment — a bipartite structure that visually surfaces which work-interference categories cluster with each treatment outcome.